# Exploração dos Microdados do ENADE 2023

## Arquivos selecionados

O pacote do INEP contém 32 arquivos TXT. Para evitar leituras desnecessárias, esta etapa utilizará inicialmente apenas:

- `microdados2023_arq1.txt`: contém `CO_CURSO`, `CO_IES`, `CO_GRUPO`, `CO_MODALIDADE` e informações de localização do curso;
- `microdados2023_arq3.txt`: contém `CO_CURSO`, `TP_PRES` e `NT_GER`.

O arquivo `Dicionário_arquivos_variáveis_microdados_Enade_2023.xlsx` será utilizado para validar o significado das variáveis e dos códigos.

## Regra de relacionamento

Por causa da anonimização aplicada pelo INEP, os arquivos não podem ser relacionados pela posição das linhas nem no nível individual do estudante.
Cada arquivo será tratado separadamente, agregado no nível de curso e relacionado exclusivamente por `CO_CURSO`, conforme orientação do manual do ENADE 2023.

## Estratégia de processamento

Para reduzir o consumo de memória e melhorar o desempenho:

- somente os arquivos necessários serão carregados;
- o parâmetro `usecols` limitará a leitura às colunas utilizadas;
- as notas serão filtradas para `TP_PRES = 555`;
- os dados serão agregados por `CO_CURSO` antes do relacionamento;
- os arquivos originais permanecerão inalterados na camada Bronze.

# 01 — Imports e caminhos dos arquivos

**Objetivo:** preparar o ambiente e definir caminhos portáteis para os
arquivos do ENADE, do Censo da Educação Superior e do banco DuckDB.

**Resultado esperado:** localizar os arquivos de origem e deixar as
bibliotecas necessárias disponíveis para as etapas seguintes.

In [1]:
%pip install pandas openpyxl duckdb -q
   
from pathlib import Path
import duckdb
import pandas as pd

##Criação de Pastas:

pasta_atual = Path.cwd()
pasta_projeto = ( pasta_atual.parent if pasta_atual.name == "notebooks" else pasta_atual )
pasta_raw = pasta_projeto / "data" / "raw"
pasta_base = ( pasta_raw / "microdados_enade_2023" / "Microdados_Enade_2023" )
pasta_dados = pasta_base / "DADOS"

## Acesso a Arquivos:
arquivo_dicionario = ( pasta_base / "1.LEIA-ME" / "Dicionário_arquivos_variáveis_microdados_Enade_2023.xlsx" )
arquivo_cursos = pasta_dados / "microdados2023_arq1.txt" 
arquivo_notas = pasta_dados / "microdados2023_arq3.txt"
arquivos_censo_cursos = list( pasta_raw.rglob("MICRODADOS_CADASTRO_CURSOS_2023.CSV") )
arquivos_censo_ies = list( pasta_raw.rglob("MICRODADOS_ED_SUP_IES_2023.CSV") )

#Validação se Arquivos foram encontrados:
if not arquivos_censo_cursos: 
    raise FileNotFoundError( "MICRODADOS_CADASTRO_CURSOS_2023.CSV não encontrado." )

if not arquivos_censo_ies:
    raise FileNotFoundError( "MICRODADOS_ED_SUP_IES_2023.CSV não encontrado." )

arquivo_cadastro_cursos = arquivos_censo_cursos[0]
arquivo_cadastro_ies = arquivos_censo_ies[0]

print(f"Dicionário encontrado: {arquivo_dicionario.exists()}")
print(f"Arquivo de cursos ENADE encontrado: {arquivo_cursos.exists()}") 
print(f"Arquivo de notas ENADE encontrado: {arquivo_notas.exists()}") 
print(f"Cadastro de cursos Censo: {arquivo_cadastro_cursos.exists()}")
print(f"Cadastro de IES Censo: {arquivo_cadastro_ies.exists()}")

Note: you may need to restart the kernel to use updated packages.
Dicionário encontrado: True
Arquivo de cursos ENADE encontrado: True
Arquivo de notas ENADE encontrado: True
Cadastro de cursos Censo: True
Cadastro de IES Censo: True


# 02 — Conferência do Dicionário de Dados

**Objetivo:** Ler o dicionário oficial dos Microdados do ENADE 2023 (aba `DICIONÁRIO_ARQUIVOS`) para identificar quais arquivos e variáveis são necessários para o desafio.

**Por quê:** Evita ler arquivos desnecessários, usar variáveis erradas, interpretar códigos incorretamente, gastar memória/tempo à toa e gera documentação da origem das variáveis escolhidas.

**Resultado esperado:** uma tabela com nome do arquivo, descrição e variáveis disponíveis, confirmando que:
- `microdados2023_arq1.txt` — dados de curso, IES, área e modalidade
- `microdados2023_arq3.txt` — dados de presença e nota geral
- variáveis necessárias: `CO_CURSO`, `CO_IES`, `CO_GRUPO`, `CO_MODALIDADE`, `TP_PRES`, `NT_GER`

In [2]:
dicionario_arquivos = pd.read_excel(
    arquivo_dicionario,
    sheet_name="DICIONÁRIO_ARQUIVOS"
)

dicionario_arquivos.head()

,Nome do arquivo,Informações,Variáveis
0,microdados2023_arq1,"Edição, código de curso e caracterização do cu...","NU_ANO, CO_CURSO, CO_IES, CO_CATEGAD, CO_ORGAC..."
1,microdados2023_arq2,"Edição, código de curso e informações acadêmic...","NU_ANO, CO_CURSO, ANO_FIM_EM, ANO_IN_GRAD, CO_..."
2,microdados2023_arq3,"Edição, código de curso e nº de itens válidos ...","NU_ANO, CO_CURSO, NU_ITEM_OFG, NU_ITEM_OFG_Z, ..."
3,microdados2023_arq4,"Edição, código de curso e avaliação dos estuda...","NU_ANO, CO_CURSO, QE_I27, QE_I28, QE_I29, QE_I..."
4,microdados2023_arq5,"Edição, código de curso e sexo","NU_ANO, CO_CURSO, TP_SEXO"


# 03 — Leitura seletiva dos dados

Os arquivos do ENADE são ordenados de maneiras diferentes e não podem
ser relacionados pela posição das linhas. Cada arquivo será tratado
separadamente e o relacionamento ocorrerá exclusivamente por
`CO_CURSO`, após a adequação para a granularidade de curso.

Para reduzir o uso de memória, somente as colunas necessárias serão
carregadas no Pandas. A camada Bronze do DuckDB continuará preservando
os arquivos completos.

In [3]:
colunas_cursos = [ "CO_CURSO", "CO_IES", "CO_GRUPO", "CO_MODALIDADE", "CO_MUNIC_CURSO", "CO_UF_CURSO" ]

colunas_notas = [ "CO_CURSO", "TP_PRES", "NT_GER" ]

cursos = pd.read_csv( arquivo_cursos, sep=";", encoding="utf-8-sig", usecols=colunas_cursos )

notas = pd.read_csv( arquivo_notas, sep=";", encoding="utf-8-sig", usecols=colunas_notas )

print(f"Linhas em cursos: {len(cursos):,}") 
print(f"Linhas em notas: {len(notas):,}") 
print(f"Cursos únicos: {cursos['CO_CURSO'].nunique():,}")

display(cursos.head()) 
display(notas.head())

Linhas em cursos: 406,294
Linhas em notas: 406,294
Cursos únicos: 9,812


,CO_CURSO,CO_IES,CO_GRUPO,CO_MODALIDADE,CO_MUNIC_CURSO,CO_UF_CURSO
0,3,1,5710,1,5103403,51
1,3,1,5710,1,5103403,51
2,3,1,5710,1,5103403,51
3,3,1,5710,1,5103403,51
4,3,1,5710,1,5103403,51


,CO_CURSO,TP_PRES,NT_GER
0,1420197,222,NaN
1,1315386,222,NaN
2,1484333,222,NaN
3,1161015,222,NaN
4,17941,222,NaN


# 04 — Definição da modelagem dimensional

A camada Gold utiliza um modelo dimensional no qual
`fato_desempenho_curso` se relaciona somente com `dim_cursos` por
`CO_CURSO`. A dimensão de cursos funciona como ponto central para as
demais dimensões.

Dimensões e relacionamentos:

- **`dim_cursos (dc)`**: código e nome do curso, além das chaves das
  demais dimensões. Relacionamento:
  `fato_desempenho_curso[CO_CURSO] <> dim_cursos[CO_CURSO]`.
- **`dim_ies (di)`**: código, nome e sigla da instituição.
  `dim_cursos[CO_IES] <> dim_ies[CO_IES]`.
- **`dim_grupo (dg)`**: código e descrição da área avaliada.
  `dim_cursos[CO_GRUPO] <> dim_grupo[CO_GRUPO]`.
- **`dim_modalidade (dmo)`**: código e descrição da modalidade.
  `dim_cursos[CO_MODALIDADE] <> dim_modalidade[CO_MODALIDADE]`.
- **`dim_localizacao (dl)`**: código e nome do município, código e
  sigla da UF. `dim_cursos[CO_MUNIC_CURSO] <>
  dim_localizacao[CO_MUNIC_CURSO]`.

Tabela fato:

- **`fato_desempenho_curso`**: nota geral média e quantidade de
  estudantes avaliados por curso.

```text
dim_ies ────────────┐
dim_grupo ──────────┤
dim_modalidade ─────┼── dim_cursos ─── fato_desempenho_curso
dim_localizacao ────┘
```

# 05 — Criação do banco e das camadas

O DuckDB armazenará as três camadas da arquitetura medalhão:

- **Bronze:** arquivos originais, sem transformação;
- **Silver:** cursos deduplicados e notas válidas;
- **Gold:** dimensões e tabela fato prontas para o dashboard.

In [4]:
pasta_banco = pasta_projeto / "data" / "database"
pasta_banco.mkdir(parents=True, exist_ok=True)

arquivo_banco = pasta_banco / "enade.duckdb"
conexao = duckdb.connect(str(arquivo_banco))

conexao.execute("CREATE SCHEMA IF NOT EXISTS bronze")
conexao.execute("CREATE SCHEMA IF NOT EXISTS silver")
conexao.execute("CREATE SCHEMA IF NOT EXISTS gold")

schemas_criados = conexao.execute(
"""
SELECT schema_name
FROM information_schema.schemata
WHERE schema_name IN ('bronze', 'silver', 'gold')
ORDER BY schema_name
"""
).df()

print(f"Banco criado em: {arquivo_banco}")
display(schemas_criados)

Banco criado em: C:\Users\vihba\OneDrive\Documents\DESAFIO UNIFOR\data\database\enade.duckdb


,schema_name
0,bronze
1,gold
2,silver


## 5.1 — Camada Bronze

A Bronze armazena os quatro arquivos de origem completos, sem filtros,
deduplicações ou tratamento de nulos. Os campos são carregados como
texto para preservar os valores recebidos das fontes oficiais.

In [5]:
##5.1.1. ARQUIVOS ENADE:

## Caminhos Para Arquivos Notas e Cursos ENADE:
caminho_cursos_bronze = arquivo_cursos.as_posix().replace("'", "''")
caminho_notas_bronze = arquivo_notas.as_posix().replace("'", "''")

## Criação ou Substituição de Tabelas ENADE:
conexao.execute(
f"""
CREATE OR REPLACE TABLE bronze.enade_cursos_raw AS
SELECT *
FROM read_csv_auto(
'{caminho_cursos_bronze}',
delim = ';',
header = true,
all_varchar = true
)
"""
)

conexao.execute(
f"""
CREATE OR REPLACE TABLE bronze.enade_notas_raw AS
SELECT *
FROM read_csv_auto(
'{caminho_notas_bronze}',
delim = ';',
header = true,
all_varchar = true
)
"""
)

print("Arquivos do ENADE gravados na Bronze.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Arquivos do ENADE gravados na Bronze.


In [6]:
##5.1.2. ARQUIVOS DO CENSO: 

## Caminhos Para Arquivos Cursos e IES Bronze:
caminho_censo_cursos_bronze = ( arquivo_cadastro_cursos.as_posix().replace("'", "''") )
caminho_censo_ies_bronze = ( arquivo_cadastro_ies.as_posix().replace("'", "''") )

## Criação ou Substituição de Tabelas ENADE:
conexao.execute(
    f""" CREATE OR REPLACE TABLE bronze.censo_cursos_raw 
    AS SELECT * FROM read_csv_auto( '{caminho_censo_cursos_bronze}', delim = ';', header = true, encoding = 'latin-1', all_varchar = true ) 
    """ )
conexao.execute( 
    f""" CREATE OR REPLACE TABLE bronze.censo_ies_raw 
    AS SELECT * FROM read_csv_auto( '{caminho_censo_ies_bronze}', delim = ';', header = true, encoding = 'latin-1', all_varchar = true ) 
    """ )

validacao_bronze = conexao.execute( 
    """ SELECT 
    'enade_cursos_raw' AS TABELA, 
    COUNT(*) AS QT_REGISTROS
        FROM bronze.enade_cursos_raw 
UNION ALL 
    SELECT 
    'enade_notas_raw', 
    COUNT(*) 
    FROM bronze.enade_notas_raw 
UNION ALL 
    SELECT 'censo_cursos_raw',
    COUNT(*) 
    FROM bronze.censo_cursos_raw 
UNION ALL 
    SELECT 'censo_ies_raw', 
    COUNT(*) 
    FROM bronze.censo_ies_raw ORDER BY TABELA """ ).df()

display(validacao_bronze)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,TABELA,QT_REGISTROS
0,censo_cursos_raw,671610
1,censo_ies_raw,2580
2,enade_cursos_raw,406294
3,enade_notas_raw,406294


## 5.2 — Camada Silver

A Silver contém os tratamentos necessários antes da agregação:

- um registro por `CO_CURSO`;
- somente estudantes presentes (`TP_PRES = 555`);
- remoção de `NT_GER` nula antes do cálculo da média.

As notas permanecem no nível original nesta camada. A agregação por
curso será feita somente na tabela fato da Gold.

In [7]:
# Seleção de colunas que identificam e caracterizam cada curso:
colunas_dimensao = [ "CO_CURSO", "CO_IES", "CO_GRUPO", "CO_MODALIDADE", "CO_MUNIC_CURSO", "CO_UF_CURSO" ]

#Verifica se um mesmo curso possui mais de um valor para algum de seus atributos
conflitos = ( cursos .groupby("CO_CURSO")[colunas_dimensao[1:]] .nunique() .gt(1) .any(axis=1) .sum() )
print(f"Cursos com informações conflitantes: {conflitos}")

# Interrompe a execução caso existam cursos inconsistentes
if conflitos != 0: 
    raise ValueError( "Existem cursos com atributos divergentes." )

Cursos com informações conflitantes: 0


In [8]:
# Cria a tabela Silver com um único registro por Curso
cursos_silver = ( cursos[colunas_dimensao] .drop_duplicates(subset=["CO_CURSO"]) .sort_values("CO_CURSO") .reset_index(drop=True) )

# Confere a unicidade da chave CO_CURSO
print( f"Cursos na Silver: {len(cursos_silver):,}" )
print( "Códigos de curso duplicados:", cursos_silver["CO_CURSO"].duplicated().sum() )

Cursos na Silver: 9,812
Códigos de curso duplicados: 0


In [9]:
# Mantém APENAS estudantes presentes e com a nota geral preenchida e reseta o Index
notas_validas = ( notas.loc[ (notas["TP_PRES"] == 555) & notas["NT_GER"].notna() ] .copy() .reset_index(drop=True) )

# Confere o impacto dos filtros aplicados
print(f"Registros originais: {len(notas):,}") 
print(f"Registros válidos: {len(notas_validas):,}") 
print( f"Registros removidos: {len(notas) - len(notas_validas):,}" ) 
print( f"Notas nulas na Silver: " f"{notas_validas['NT_GER'].isna().sum():,}" )

Registros originais: 406,294
Registros válidos: 346,519
Registros removidos: 59,775
Notas nulas na Silver: 0


In [10]:
conexao.register("df_cursos_silver", cursos_silver)

conexao.execute(
"""
CREATE OR REPLACE TABLE silver.cursos_tratados AS
SELECT * FROM df_cursos_silver
"""
)

conexao.unregister("df_cursos_silver")

conexao.register("df_notas_validas", notas_validas)

conexao.execute(
"""
CREATE OR REPLACE TABLE silver.notas_validas AS
SELECT * FROM df_notas_validas
"""
)

conexao.unregister("df_notas_validas")

validacao_silver = conexao.execute(
"""
SELECT
'cursos_tratados' AS TABELA,
COUNT(*) AS QT_REGISTROS,
COUNT(*) - COUNT(DISTINCT CO_CURSO) AS QT_DUPLICADOS,
NULL AS QT_NOTAS_NULAS
FROM silver.cursos_tratados

UNION ALL

SELECT
'notas_validas',
COUNT(*),
NULL,
SUM(CASE WHEN NT_GER IS NULL THEN 1 ELSE 0 END)
FROM silver.notas_validas
"""
).df()

display(validacao_silver)

,TABELA,QT_REGISTROS,QT_DUPLICADOS,QT_NOTAS_NULAS
0,cursos_tratados,9812,0,NaN
1,notas_validas,346519,<NA>,0.0


## 5.3 — Camada Gold

A Gold contém o modelo dimensional definitivo. As dimensões armazenam
os atributos descritivos e a fato armazena somente as métricas por
curso. Não será criada uma tabela analítica duplicando esses atributos.

In [11]:
# Disponibiliza o DataFrame de cursos temporariamente no DuckDB
conexao.register( "df_cursos_silver", cursos_silver )

# Cria ou substitui a tabela física de cursos na Silver
conexao.execute( """ CREATE OR REPLACE TABLE silver.cursos_tratados AS SELECT * FROM df_cursos_silver """ )
conexao.unregister( "df_cursos_silver" )

# Disponibiliza o DataFrame de notas temporariamente no DuckDB
conexao.register( "df_notas_validas", notas_validas )

# Cria ou substitui a tabela física de notas na Silver
conexao.execute( """ CREATE OR REPLACE TABLE silver.notas_validas AS SELECT * FROM df_notas_validas """ )
conexao.unregister( "df_notas_validas" )

# Verifica a quantidade de registros, cursos duplicados e notas nulas nas tabelas armazenadas na Silver:

validacao_silver = conexao.execute( 
""" 
    SELECT 
        'cursos_tratados' AS TABELA,
        COUNT(*) AS QT_REGISTROS,
        COUNT(*) - COUNT(DISTINCT CO_CURSO) AS QT_DUPLICADOS,
        NULL AS QT_NOTAS_NULAS 
    FROM silver.cursos_tratados
UNION ALL
    SELECT 
        'notas_validas' AS TABELA, 
        COUNT(*) AS QT_REGISTROS, 
        NULL AS QT_DUPLICADOS, 
        SUM( CASE WHEN NT_GER IS NULL THEN 1 ELSE 0 END ) AS QT_NOTAS_NULAS 
    FROM silver.notas_validas 
""" ).df()

display(validacao_silver)

,TABELA,QT_REGISTROS,QT_DUPLICADOS,QT_NOTAS_NULAS
0,cursos_tratados,9812,0,NaN
1,notas_validas,346519,<NA>,0.0


### 5.3.1. Criação de Dimensões:

#### 5.3.1.1. Dimensão Cursos (dim_cursos):

In [12]:
## Extrair, Deduplicar e Padronizar da Dimensão Cursos do Censo:
cadastro_cursos = pd.read_csv( arquivo_cadastro_cursos, sep=";", encoding="latin-1", usecols=["CO_CURSO", "NO_CURSO"] )
cadastro_cursos = ( cadastro_cursos .dropna(subset=["CO_CURSO"]) .drop_duplicates(subset=["CO_CURSO"]) )

## Mapear de Nomes dos Cursos no ENADE via CO_CURSO
dim_cursos = ( cursos_silver .merge( cadastro_cursos, on="CO_CURSO", how="left", validate="one_to_one" ) [[ "CO_CURSO", "NO_CURSO", "CO_IES", "CO_GRUPO", "CO_MODALIDADE", "CO_MUNIC_CURSO", "CO_UF_CURSO" ]] .sort_values("CO_CURSO") .reset_index(drop=True) )

display(dim_cursos.head())

,CO_CURSO,NO_CURSO,CO_IES,CO_GRUPO,CO_MODALIDADE,CO_MUNIC_CURSO,CO_UF_CURSO
0,3,Engenharia Civil,1,5710,1,5103403,51
1,9,Agronomia,1,17,1,5103403,51
2,10,Engenharia Florestal,1,6405,1,5103403,51
3,12,Medicina,1,12,1,5103403,51
4,16,Engenharia Elétrica,1,5806,1,5103403,51


#### 5.3.1.2. Dimensão IES (dim_ies):

In [13]:
# Seleciona os campos relevantes da IES
cadastro_ies = pd.read_csv( arquivo_cadastro_ies, sep=";", encoding="latin-1", usecols=[ "CO_IES", "NO_IES", "SG_IES" ] )

# Remove registros sem código e mantém uma linha por instituição
cadastro_ies = ( cadastro_ies .dropna(subset=["CO_IES"]) .drop_duplicates(subset=["CO_IES"]) )

# Seleciona as IES presentes no ENADE e acrescenta nome e sigla

dim_ies = ( dim_cursos[["CO_IES"]] .drop_duplicates() .merge( cadastro_ies, on="CO_IES", how="left", validate="one_to_one" ) [[ "CO_IES", "NO_IES", "SG_IES" ]] .sort_values("CO_IES") .reset_index(drop=True) )

# Exibe uma amostra da dimensão criada

display(dim_ies.head())

,CO_IES,NO_IES,SG_IES
0,1,UNIVERSIDADE FEDERAL DE MATO GROSSO,UFMT
1,2,UNIVERSIDADE DE BRASÍLIA,UNB
2,3,UNIVERSIDADE FEDERAL DE SERGIPE,UFS
3,4,UNIVERSIDADE FEDERAL DO AMAZONAS,UFAM
4,5,UNIVERSIDADE FEDERAL DO PIAUÍ,UFPI


### 5.3.1.3. Dimensão grupo (dim_grupo):

In [14]:
# Lê no dicionário ENADE:
mapa_grupos = pd.read_excel( arquivo_dicionario, sheet_name="DICIONÁRIO DE VARIÁVEIS", usecols="E", skiprows=27, nrows=28, header=None, names=["MAPEAMENTO"] )

# Divide um o campo concatenado em dois outras colunas:
mapa_grupos[["CO_GRUPO", "NO_GRUPO"]] = ( mapa_grupos["MAPEAMENTO"] .str.split("=", n=1, expand=True) )
mapa_grupos["CO_GRUPO"] = pd.to_numeric( mapa_grupos["CO_GRUPO"].str.strip(), errors="coerce" ).astype("Int64")
mapa_grupos["NO_GRUPO"] = ( mapa_grupos["NO_GRUPO"] .str.strip() )

# Mantém somente códigos válidos e únicos

mapa_grupos = ( mapa_grupos[["CO_GRUPO", "NO_GRUPO"]] .dropna(subset=["CO_GRUPO"]) .drop_duplicates(subset=["CO_GRUPO"]) )

# Seleciona os grupos presentes no ENADE e acrescenta suas descrições oficiais

dim_grupo = ( dim_cursos[["CO_GRUPO"]] .drop_duplicates() .merge( mapa_grupos, on="CO_GRUPO", how="left", validate="one_to_one" ) [[ "CO_GRUPO", "NO_GRUPO" ]] .sort_values("CO_GRUPO") .reset_index(drop=True) )
print( "Grupos sem descrição:", dim_grupo["NO_GRUPO"].isna().sum() )

dim_grupo.head(5)

Grupos sem descrição: 0


,CO_GRUPO,NO_GRUPO
0,5,Medicina Veterinária
1,6,Odontologia
2,12,Medicina
3,17,Agronomia
4,19,Farmácia


#### 5.3.1.4. Dimensão Modalidade (dim_modalidade):

In [15]:
# Remove duplicada de CO_MODALIDADE:
dim_modalidade = ( dim_cursos[["CO_MODALIDADE"]] .drop_duplicates() .sort_values("CO_MODALIDADE") .reset_index(drop=True) )

# Cria coluna DS_MODALIDADE com a escrição de EaD ou Presencial
dim_modalidade["DS_MODALIDADE"] = ( dim_modalidade["CO_MODALIDADE"] .map({ 0: "EaD", 1: "Presencial" }) )
print( "Modalidades sem descrição:", dim_modalidade["DS_MODALIDADE"].isna().sum() )

display(dim_modalidade)

Modalidades sem descrição: 0


,CO_MODALIDADE,DS_MODALIDADE
0,0,EaD
1,1,Presencial


#### 5.3.1.5. Dimensão Localização (dim_localizacao):

In [16]:
# Lê no dicionário ENADE:

cadastro_municipios = pd.read_excel( arquivo_dicionario, sheet_name="MUNICÍPIOS", skiprows=3, usecols="B:D" )

# Padroniza os nomes das colunas para o modelo dimensional
cadastro_municipios = cadastro_municipios.rename( columns={ "CÓDIGO DO MUNICÍPIO": "CO_MUNIC_CURSO", "NOME DO MUNICÍPIO": "NO_MUNIC_CURSO", "UF": "SG_UF" } )
cadastro_municipios["CO_MUNIC_CURSO"] = pd.to_numeric( cadastro_municipios["CO_MUNIC_CURSO"], errors="coerce" ).astype("Int64")

# Remove códigos inválidos e mantém um registro por município
cadastro_municipios = ( cadastro_municipios .dropna(subset=["CO_MUNIC_CURSO"]) .drop_duplicates(subset=["CO_MUNIC_CURSO"]) )

# Cria dim_localização trazendo a informações de município e UF.

dim_localizacao = ( dim_cursos[[ "CO_MUNIC_CURSO", "CO_UF_CURSO" ]] .drop_duplicates(subset=["CO_MUNIC_CURSO"]) .merge( cadastro_municipios[[ "CO_MUNIC_CURSO", "NO_MUNIC_CURSO", "SG_UF" ]], on="CO_MUNIC_CURSO", how="left", validate="one_to_one" ) [[ "CO_MUNIC_CURSO", "NO_MUNIC_CURSO", "CO_UF_CURSO", "SG_UF" ]] .sort_values("CO_MUNIC_CURSO") .reset_index(drop=True) )
print( "Municípios sem nome:", dim_localizacao["NO_MUNIC_CURSO"].isna().sum() )
print( "Localizações sem sigla da UF:", dim_localizacao["SG_UF"].isna().sum() )

dim_localizacao.head(5)

Municípios sem nome: 0
Localizações sem sigla da UF: 0


,CO_MUNIC_CURSO,NO_MUNIC_CURSO,CO_UF_CURSO,SG_UF
0,1100023,ARIQUEMES,11,RO
1,1100049,CACOAL,11,RO
2,1100064,COLORADO DO OESTE,11,RO
3,1100114,JARU,11,RO
4,1100122,JI-PARANA,11,RO


### 5.3.2. Criação da Tabela Fato (fato_desempenho_curso)

In [17]:
##5.3.2.1. Agregação e Cálculo de Métricas de Desempenho por Curso
fato_desempenho_curso = ( notas_validas .groupby( "CO_CURSO", as_index=False ) .agg( NT_GER_MEDIA=("NT_GER", "mean"), QT_AVALIADOS=("NT_GER", "size") ) )
fato_desempenho_curso["NT_GER_MEDIA"] = ( fato_desempenho_curso["NT_GER_MEDIA"] .round(2) )

print( f"notas_validas: " f"{len(notas_validas):,}" )
fato_desempenho_curso.head(5)

notas_validas: 346,519


,CO_CURSO,NT_GER_MEDIA,QT_AVALIADOS
0,3,59.61,31
1,9,59.38,36
2,10,45.71,11
3,12,69.44,78
4,16,51.27,23


In [18]:
##5.3.2.2. Validar unicidade de chaves e integridade relacional Gold

##Validação de Unicidade da Chave Primária de Cursos
if not dim_cursos["CO_CURSO"].is_unique: 
    raise ValueError( "dim_cursos possui CO_CURSO duplicado." )

## Verificação de Duplicatas e Consistência fato_desempenho_curso vs dim_cursos
if not fato_desempenho_curso["CO_CURSO"].is_unique: 
    raise ValueError( "fato_desempenho_curso possui CO_CURSO duplicado." )

cursos_sem_dimensao = fato_desempenho_curso.loc[ ~fato_desempenho_curso["CO_CURSO"].isin( dim_cursos["CO_CURSO"] ) ]

if not cursos_sem_dimensao.empty: 
    raise ValueError( f"{len(cursos_sem_dimensao)} cursos da fato " "não existem em dim_cursos." )

print("Tabelas Gold criadas:") 
print(f"dim_cursos: {len(dim_cursos):,}") 
print(f"dim_ies: {len(dim_ies):,}") 
print(f"dim_grupo: {len(dim_grupo):,}") 
print(f"dim_modalidade: {len(dim_modalidade):,}") 
print(f"dim_localizacao: {len(dim_localizacao):,}") 
print( f"fato_desempenho_curso: " f"{len(fato_desempenho_curso):,}" )
print( "Cursos da fato sem dimensão:", len(cursos_sem_dimensao) )

Tabelas Gold criadas:
dim_cursos: 9,812
dim_ies: 1,347
dim_grupo: 28
dim_modalidade: 2
dim_localizacao: 718
fato_desempenho_curso: 9,380
Cursos da fato sem dimensão: 0


### 5.3.3. Associação de DataFrames Pandas às Tabelas Gold

In [19]:
##5.3.3.1. Relaciona o nome de cada tabela Gold # ao respectivo DataFrame do Pandas

tabelas_gold = { "dim_cursos": dim_cursos, "dim_ies": dim_ies, "dim_grupo": dim_grupo, "dim_modalidade": dim_modalidade, "dim_localizacao": dim_localizacao, "fato_desempenho_curso": fato_desempenho_curso }

##5.3.3.2. Registrar os DataFrames no DuckDB e materializar as tabelas físicas no schema Gold.

for nome_tabela, dataframe in tabelas_gold.items(): 
    nome_temporario = f"df_{nome_tabela}"
    conexao.register(nome_temporario, dataframe)
    conexao.execute( f""" CREATE OR REPLACE TABLE gold.{nome_tabela} AS SELECT * FROM {nome_temporario} """ )
    conexao.unregister( nome_temporario )

print("Modelo dimensional gravado na camada Gold")

Modelo dimensional gravado na camada Gold


In [20]:
##5.3.3.3. Validar a volumetria e quantidade de registros gravados no schema Gold

validacao_gold = conexao.execute(
"""
SELECT 'dim_cursos' AS TABELA, COUNT(*) AS QT_REGISTROS
FROM gold.dim_cursos
UNION ALL
SELECT 'dim_ies', COUNT(*) FROM gold.dim_ies
UNION ALL
SELECT 'dim_grupo', COUNT(*) FROM gold.dim_grupo
UNION ALL
SELECT 'dim_modalidade', COUNT(*) FROM gold.dim_modalidade
UNION ALL
SELECT 'dim_localizacao', COUNT(*) FROM gold.dim_localizacao
UNION ALL
SELECT 'fato_desempenho_curso', COUNT(*)
FROM gold.fato_desempenho_curso
ORDER BY TABELA
"""
).df()

##5.3.3.4. Validação de Integridade Referencial entre Fato e Dimensã
integridade_gold = conexao.execute(
"""
SELECT COUNT(*) AS CURSOS_DA_FATO_SEM_DIMENSAO
FROM gold.fato_desempenho_curso AS f
LEFT JOIN gold.dim_cursos AS d
ON f.CO_CURSO = d.CO_CURSO
WHERE d.CO_CURSO IS NULL
"""
).df()

display(validacao_gold)
display(integridade_gold)

,TABELA,QT_REGISTROS
0,dim_cursos,9812
1,dim_grupo,28
2,dim_ies,1347
3,dim_localizacao,718
4,dim_modalidade,2
5,fato_desempenho_curso,9380


,CURSOS_DA_FATO_SEM_DIMENSAO
0,0


# 06 —  Respostas às Perguntas de Negócio com SQL


As perguntas do desafio serão respondidas por consultas SQL executadas diretamente sobre o modelo dimensional da camada Gold.

### 6.1 — A Unifor está no ENADE 2023?

In [21]:
resumo_unifor = conexao.execute(
"""
SELECT
i.CO_IES,
i.NO_IES,
i.SG_IES,
COUNT(DISTINCT c.CO_CURSO) AS QT_CURSOS,
COUNT(DISTINCT c.CO_GRUPO) AS QT_AREAS,
COUNT(DISTINCT c.CO_MODALIDADE) AS QT_MODALIDADES
FROM gold.dim_ies AS i
INNER JOIN gold.dim_cursos AS c
    ON i.CO_IES = c.CO_IES

WHERE
    UPPER(TRIM(i.SG_IES)) = 'UNIFOR'
    OR UPPER(i.NO_IES) LIKE '%UNIVERSIDADE DE FORTALEZA%'

GROUP BY
i.CO_IES,
i.NO_IES,
i.SG_IES
"""
).df()

display(resumo_unifor)

if resumo_unifor.empty:
    print("Não. A Unifor não está presente no ENADE 2023.")
else:
    linha = resumo_unifor.iloc[0]

    print(
        f"Sim. A Unifor está presente no ENADE 2023, "
        f"com {int(linha['QT_CURSOS'])} cursos e "
        f"{int(linha['QT_MODALIDADES'])} modalidades."
    )

,CO_IES,NO_IES,SG_IES,QT_CURSOS,QT_AREAS,QT_MODALIDADES
0,555,UNIVERSIDADE DE FORTALEZA,UNIFOR,17,17,1


Sim. A Unifor está presente no ENADE 2023, com 17 cursos e 1 modalidades.


In [22]:
cursos_unifor = conexao.execute("""
SELECT DISTINCT
    c.NO_CURSO
FROM gold.dim_cursos AS c
INNER JOIN gold.dim_ies AS i
    ON c.CO_IES = i.CO_IES
WHERE
    UPPER(TRIM(i.SG_IES)) = 'UNIFOR'
    OR UPPER(i.NO_IES) LIKE '%UNIVERSIDADE DE FORTALEZA%'
ORDER BY
    c.NO_CURSO
""").df()

print("Cursos que a Unifor possui:")

if cursos_unifor.empty:
    print("Nenhum curso encontrado.")
else:
    for numero, curso in enumerate(cursos_unifor["NO_CURSO"], start=1):
        print(f"{numero}. {curso}")



Cursos que a Unifor possui:
1. Arquitetura E Urbanismo
2. Enfermagem
3. Engenharia Ambiental E Sanitária
4. Engenharia Civil
5. Engenharia De Computação
6. Engenharia De Controle E Automação
7. Engenharia De Produção
8. Engenharia Elétrica
9. Engenharia Mecânica
10. Estética E Cosmética
11. Farmácia
12. Fisioterapia
13. Fonoaudiologia
14. Medicina
15. Medicina Veterinária
16. Nutrição
17. Odontologia


In [23]:
areas_modalidades_unifor = conexao.execute(
"""
SELECT
g.CO_GRUPO,
g.NO_GRUPO,
m.DS_MODALIDADE,
COUNT(DISTINCT c.CO_CURSO) AS QT_CURSOS
FROM gold.dim_cursos AS c
INNER JOIN gold.dim_ies AS i
    ON c.CO_IES = i.CO_IES
INNER JOIN gold.dim_grupo AS g
    ON c.CO_GRUPO = g.CO_GRUPO
INNER JOIN gold.dim_modalidade AS m
    ON c.CO_MODALIDADE = m.CO_MODALIDADE

WHERE
    UPPER(TRIM(i.SG_IES)) = 'UNIFOR'
    OR UPPER(i.NO_IES) LIKE '%UNIVERSIDADE DE FORTALEZA%'

GROUP BY
g.CO_GRUPO,
g.NO_GRUPO,
m.DS_MODALIDADE

ORDER BY
g.NO_GRUPO,
m.DS_MODALIDADE
"""
).df()

display(areas_modalidades_unifor)

,CO_GRUPO,NO_GRUPO,DS_MODALIDADE,QT_CURSOS
0,21,Arquitetura e Urbanismo,Presencial,1
1,23,Enfermagem,Presencial,1
2,6307,Engenharia Ambiental,Presencial,1
3,5710,Engenharia Civil,Presencial,1
4,5806,Engenharia Elétrica,Presencial,1
5,5902,Engenharia Mecânica,Presencial,1
6,6411,Engenharia de Computação I,Presencial,1
7,5814,Engenharia de Controle e Automação,Presencial,1
8,6208,Engenharia de Produção,Presencial,1
9,19,Farmácia,Presencial,1


### 6.2 — A nota média difere entre Presencial e EaD? E na UNIFOR?

In [30]:
## Comparação entre TODAS as IES:

comparacao_modalidades = conexao.execute(
"""
SELECT
m.DS_MODALIDADE,
COUNT(DISTINCT f.CO_CURSO) AS QT_CURSOS,
SUM(f.QT_AVALIADOS) AS QT_AVALIADOS,
ROUND(SUM(f.NT_GER_MEDIA * f.QT_AVALIADOS)/NULLIF(SUM(f.QT_AVALIADOS),0),2) AS NT_GER_MEDIA
FROM gold.fato_desempenho_curso AS f
INNER JOIN gold.dim_cursos AS c
    ON f.CO_CURSO = c.CO_CURSO
INNER JOIN gold.dim_modalidade AS m
    ON c.CO_MODALIDADE = m.CO_MODALIDADE

GROUP BY m.DS_MODALIDADE
ORDER BY NT_GER_MEDIA DESC
"""
).df()

display(comparacao_modalidades)

presencial = comparacao_modalidades[
    comparacao_modalidades["DS_MODALIDADE"]
    .str.upper()
    .str.contains("PRESENCIAL")
]["NT_GER_MEDIA"].iloc[0]

ead = comparacao_modalidades[
    comparacao_modalidades["DS_MODALIDADE"]
    .str.upper()
    .str.contains("EAD|DIST")
]["NT_GER_MEDIA"].iloc[0]

diferenca = abs(presencial - ead)

if diferenca > 0:
    print(f"Sim. As notas diferem em {diferenca:.2f} pontos. \n \n \n")
else:
    print("Não. As notas são iguais.")

## Teste: a UNIFOR possui cursos nas modalidades Presencial e EAD?

teste_unifor = conexao.execute(
    """
    SELECT
        i.NO_IES,
        m.DS_MODALIDADE,
        COUNT(DISTINCT f.CO_CURSO) AS QT_CURSOS,
        SUM(f.QT_AVALIADOS) AS QT_AVALIADOS,
        ROUND(SUM(f.NT_GER_MEDIA * f.QT_AVALIADOS) / NULLIF(SUM(f.QT_AVALIADOS), 0), 2) AS NT_GER_MEDIA
    FROM gold.fato_desempenho_curso AS f
    INNER JOIN gold.dim_cursos AS c
        ON f.CO_CURSO = c.CO_CURSO
    INNER JOIN gold.dim_modalidade AS m
        ON c.CO_MODALIDADE = m.CO_MODALIDADE
    INNER JOIN gold.dim_ies AS i
        ON c.CO_IES = i.CO_IES
    WHERE UPPER(i.NO_IES) LIKE '%UNIFOR%'
       OR UPPER(i.SG_IES) = 'UNIFOR'
    GROUP BY i.NO_IES, m.DS_MODALIDADE
    ORDER BY m.DS_MODALIDADE
    """
).df()

display(teste_unifor)

modalidades_unifor = set(teste_unifor["DS_MODALIDADE"].str.upper())

tem_presencial = any("PRESENCIAL" in mod for mod in modalidades_unifor)
tem_ead = any(("EAD" in mod) or ("DIST" in mod) for mod in modalidades_unifor)

if tem_presencial and tem_ead:
    print("Sim, a UNIFOR possui cursos tanto na modalidade Presencial quanto na EAD.")
elif tem_presencial and not tem_ead:
    print("A UNIFOR possui cursos apenas na modalidade Presencial (não foram encontrados cursos EAD).")
elif tem_ead and not tem_presencial:
    print("A UNIFOR possui cursos apenas na modalidade EAD (não foram encontrados cursos Presenciais).")
else:
    print("Não foram encontrados cursos da UNIFOR no banco de dados.")


,DS_MODALIDADE,QT_CURSOS,QT_AVALIADOS,NT_GER_MEDIA
0,Presencial,8757,300277.0,49.73
1,EaD,623,46242.0,38.90


Sim. As notas diferem em 10.83 pontos. 
 
 



,NO_IES,DS_MODALIDADE,QT_CURSOS,QT_AVALIADOS,NT_GER_MEDIA
0,UNIVERSIDADE DE FORTALEZA,Presencial,17,1235.0,53.52


A UNIFOR possui cursos apenas na modalidade Presencial (não foram encontrados cursos EAD).


### 6.3 — Quais são os dez cursos da Unifor com maior nota média? E na UNIFOR?

In [25]:
top_10_cursos_unifor = conexao.execute(
"""
SELECT
c.CO_CURSO,
i.SG_IES,
c.NO_CURSO,
g.NO_GRUPO,
m.DS_MODALIDADE,
ROUND(f.NT_GER_MEDIA, 2) AS NT_GER_MEDIA,
f.QT_AVALIADOS
FROM gold.fato_desempenho_curso AS f
INNER JOIN gold.dim_cursos AS c
    ON f.CO_CURSO = c.CO_CURSO
INNER JOIN gold.dim_ies AS i
    ON c.CO_IES = i.CO_IES
INNER JOIN gold.dim_grupo AS g
    ON c.CO_GRUPO = g.CO_GRUPO
INNER JOIN gold.dim_modalidade AS m
    ON c.CO_MODALIDADE = m.CO_MODALIDADE

WHERE
    UPPER(TRIM(i.SG_IES)) = 'UNIFOR'
    OR UPPER(i.NO_IES) LIKE '%UNIVERSIDADE DE FORTALEZA%'

ORDER BY
f.NT_GER_MEDIA DESC,
f.QT_AVALIADOS DESC,
c.CO_CURSO

LIMIT 10
"""
).df()

print("Segue abaixo o TOP 10 de cursos da UNIFOR pela nota média do ENADE:\n")

display(top_10_cursos_unifor)

Segue abaixo o TOP 10 de cursos da UNIFOR pela nota média do ENADE:



,CO_CURSO,SG_IES,NO_CURSO,NO_GRUPO,DS_MODALIDADE,NT_GER_MEDIA,QT_AVALIADOS
0,93001,UNIFOR,Medicina,Medicina,Presencial,68.89,196
1,11719,UNIFOR,Enfermagem,Enfermagem,Presencial,60.98,57
2,18324,UNIFOR,Arquitetura E Urbanismo,Arquitetura e Urbanismo,Presencial,58.92,144
3,11718,UNIFOR,Fisioterapia,Fisioterapia,Presencial,57.74,52
4,18325,UNIFOR,Farmácia,Farmácia,Presencial,54.19,42
5,11731,UNIFOR,Odontologia,Odontologia,Presencial,52.95,149
6,1315325,UNIFOR,Estética E Cosmética,Tecnologia em Estética e Cosmética,Presencial,51.94,23
7,56630,UNIFOR,Nutrição,Nutrição,Presencial,51.85,93
8,1357703,UNIFOR,Medicina Veterinária,Medicina Veterinária,Presencial,51.30,92
9,107686,UNIFOR,Engenharia Ambiental E Sanitária,Engenharia Ambiental,Presencial,51.07,16


### 6.4  — Quem é a IES com a melhor nota do Brasil nas áreas em que a Unifor atua, e quão perto/longe ela está?

In [26]:
## Comparativo da Unifor vs. Instituições de Similaridade Curricular/Áreas ENADE

comparacao_ies = conexao.execute(
"""
SELECT
i.CO_IES,
i.NO_IES,
i.SG_IES,
ROUND(
SUM(f.NT_GER_MEDIA * f.QT_AVALIADOS) / SUM(f.QT_AVALIADOS),2) AS NT_GER_MEDIA,
SUM(f.QT_AVALIADOS) AS QT_AVALIADOS,
COUNT(DISTINCT c.CO_GRUPO) AS QT_AREAS
FROM gold.fato_desempenho_curso AS f
INNER JOIN gold.dim_cursos AS c
    ON f.CO_CURSO = c.CO_CURSO
INNER JOIN gold.dim_ies AS i
    ON c.CO_IES = i.CO_IES

WHERE c.CO_GRUPO IN (
    SELECT DISTINCT c2.CO_GRUPO 
    FROM gold.dim_cursos AS c2
    INNER JOIN gold.dim_ies AS i2
        ON c2.CO_IES = i2.CO_IES

WHERE i2.SG_IES = 'UNIFOR'
)

GROUP BY
i.CO_IES,
i.NO_IES,
i.SG_IES

HAVING COUNT(DISTINCT c.CO_GRUPO) >= 16

ORDER BY NT_GER_MEDIA DESC
"""
).df()

display(comparacao_ies)

## Fazer Comparativo:
comparacao_ies["POSICAO"] = (
    comparacao_ies["NT_GER_MEDIA"]
    .rank(method="min", ascending=False)
    .astype(int)
)

unifor = comparacao_ies[
    comparacao_ies["SG_IES"].str.upper().str.strip() == "UNIFOR"
]

if unifor.empty:
    print("A Unifor não foi encontrada no ranking.")
else:
    posicao = unifor.iloc[0]["POSICAO"]

    print(
        f"A Unifor está na posição {posicao} "
        f"do ranking, considerando a nota média do ENADE, "
        f"da maior para a menor."
    )

,CO_IES,NO_IES,SG_IES,NT_GER_MEDIA,QT_AVALIADOS,QT_AREAS
0,578,UNIVERSIDADE FEDERAL DA BAHIA,UFBA,60.88,1258.0,16
1,585,UNIVERSIDADE FEDERAL DE SANTA CATARINA,UFSC,60.56,1318.0,16
2,582,UNIVERSIDADE FEDERAL DE SANTA MARIA,UFSM,58.91,917.0,16
3,555,UNIVERSIDADE DE FORTALEZA,UNIFOR,53.52,1235.0,17
4,13,UNIVERSIDADE DE CAXIAS DO SUL,UCS,51.21,718.0,16
5,150,UNIVERSIDADE DE SOROCABA,UNISO,49.22,538.0,16
6,496,UNIVERSIDADE DE FRANCA,UNIFRAN,44.20,1400.0,16


A Unifor está na posição 4 do ranking, considerando a nota média do ENADE, da maior para a menor.


In [27]:
print("Por Questão de Regionalidade, irei comparar com UFBA")

# Comparando com a UFBA:

liderUFBA = (
comparacao_ies.loc[
comparacao_ies["SG_IES"] != "UNIFOR"
]
.iloc[0]
)

# Seleciona o resultado da Unifor

resultado_unifor = (
comparacao_ies.loc[
comparacao_ies["SG_IES"] == "UNIFOR"
]
.iloc[0]
)

# Calcula a distância entre as médias

distancia = round(
liderUFBA["NT_GER_MEDIA"]
- resultado_unifor["NT_GER_MEDIA"],
2
)

print(f"IES líder: {liderUFBA['NO_IES']}")
print(f"Média da líder: {liderUFBA['NT_GER_MEDIA']}")
print(f"Média da Unifor: {resultado_unifor['NT_GER_MEDIA']}")
print("A UNIFOR atingiu "+f"Distância: {distancia} pontos a menos que a UFBA")

Por Questão de Regionalidade, irei comparar com UFBA
IES líder: UNIVERSIDADE FEDERAL DA BAHIA
Média da líder: 60.88
Média da Unifor: 53.52
A UNIFOR atingiu Distância: 7.36 pontos a menos que a UFBA


In [28]:
##06 — Curiosidade opcional: Unifor x melhor IES do Brasil, área por área

curiosidade_unifor_vs_brasil = conexao.execute(
    """
    WITH areas_unifor AS (
        SELECT DISTINCT CO_GRUPO
        FROM gold.dim_cursos
        WHERE CO_IES = 555
    ),
    ranking_area AS (
        SELECT
            dc.CO_GRUPO,
            dc.CO_IES,
            f.NT_GER_MEDIA,
            RANK() OVER (PARTITION BY dc.CO_GRUPO ORDER BY f.NT_GER_MEDIA DESC) AS POSICAO,
            COUNT(*) OVER (PARTITION BY dc.CO_GRUPO) AS TOTAL_CURSOS_AREA
        FROM gold.fato_desempenho_curso f
        JOIN gold.dim_cursos dc ON f.CO_CURSO = dc.CO_CURSO
        WHERE dc.CO_GRUPO IN (SELECT CO_GRUPO FROM areas_unifor)
    ),
    melhor_do_brasil AS (
        SELECT
            r.CO_GRUPO,
            r.NT_GER_MEDIA AS MELHOR_NOTA,
            di.NO_IES AS MELHOR_IES,
            di.SG_IES AS MELHOR_SIGLA
        FROM ranking_area r
        JOIN gold.dim_ies di ON r.CO_IES = di.CO_IES
        WHERE r.POSICAO = 1
    ),
    unifor AS (
        SELECT CO_GRUPO, NT_GER_MEDIA AS NOTA_UNIFOR, POSICAO, TOTAL_CURSOS_AREA
        FROM ranking_area
        WHERE CO_IES = 555
    )
    SELECT
        dg.NO_GRUPO AS AREA,
        u.NOTA_UNIFOR,
        m.MELHOR_NOTA,
        ROUND(m.MELHOR_NOTA - u.NOTA_UNIFOR, 2) AS DIFERENCA,
        m.MELHOR_SIGLA AS MELHOR_IES_SIGLA,
        u.POSICAO AS POSICAO_UNIFOR_NACIONAL,
        u.TOTAL_CURSOS_AREA,
        ROUND(100.0 * (1 - (u.POSICAO - 1.0) / u.TOTAL_CURSOS_AREA), 1) AS PERCENTIL_TOP
    FROM unifor u
    JOIN melhor_do_brasil m ON u.CO_GRUPO = m.CO_GRUPO
    JOIN gold.dim_grupo dg ON u.CO_GRUPO = dg.CO_GRUPO
    ORDER BY DIFERENCA ASC
    """
).df()

display(curiosidade_unifor_vs_brasil)

,AREA,NOTA_UNIFOR,MELHOR_NOTA,DIFERENCA,MELHOR_IES_SIGLA,POSICAO_UNIFOR_NACIONAL,TOTAL_CURSOS_AREA,PERCENTIL_TOP
0,Medicina,68.89,77.73,8.84,FAMERP,102,305,66.9
1,Farmácia,54.19,67.35,13.16,UEFS,94,578,83.9
2,Enfermagem,60.98,74.72,13.74,UNIFESP,104,961,89.3
3,Medicina Veterinária,51.30,66.89,15.59,UFPR,101,359,72.1
4,Arquitetura e Urbanismo,58.92,74.57,15.65,UFOP,129,551,76.8
5,Odontologia,52.95,68.84,15.89,UFPI,120,448,73.4
6,Tecnologia em Estética e Cosmética,51.94,68.10,16.16,UNESC,38,271,86.3
7,Fonoaudiologia,44.67,61.43,16.76,UFMG,39,73,47.9
8,Nutrição,51.85,68.70,16.85,UNESULBAHIA,188,601,68.9
9,Fisioterapia,57.74,74.81,17.07,UFJF,186,699,73.5


In [29]:
# ## FINALIZAR 
# conexao.close()
# print("Conexão com o DuckDB encerrada.")